[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_84_Chunking_Strategies.ipynb)

# Lesson 84 — Chunking Strategies: how you cut the docs sets the ceiling

**Phase 10 · RAG at Production Scale · Lesson 2 of ~7**

Last lesson (L83) built the full naive RAG pipeline and MEASURED it. The very last
experiment was a cliffhanger: you swept `max_chars` over `{60, 120, 180, 320, ∞}`,
kept the corpus, the retriever, and the questions *identical*, and watched `hit@1`
swing anyway. That was **Failure Mode #4** — chunk size is a hyperparameter — and we
labelled it "fix this in Lesson 84." This is Lesson 84.

**The one idea:** *chunking is the only RAG decision that sets a hard ceiling on
everything downstream.* The retriever can only rank chunks that exist; a reranker
(L86) can only reorder chunks that exist; the LLM can only read chunks that were
retrieved. **If a fact was fragmented across two chunks — or drowned inside one huge
chunk — at index time, then no embedding model, no reranker, and no bigger LLM can
ever recover it.** Chunking happens once, up front, for pennies, and it silently caps
the quality of the most expensive stages you'll build later. That makes it the
highest-leverage, most-overlooked lever in the whole pipeline.

Today you implement **five** chunking strategies behind one interface, run them all
against the exact Nimbus eval set from L83, and see — with numbers — which cuts help,
which hurt, and *why the fanciest one loses when you don't tune it.*

### Where we are in Phase 10

| # | Lesson | The failure mode it kills |
|---|--------|---------------------------|
| L83 | RAG production baseline + eval harness | (built the measuring stick) |
| **L84** | **Chunking strategies ← you are here** | **FM4: chunk size / boundaries** |
| L85 | Dense & hybrid retrieval | FM2: semantic gap (lexical blindness) |
| L86 | Reranking | FM1: right chunk, wrong rank |
| L87 | Grounding & abstention | FM3: answering out-of-scope |
| L88 | RAG evaluation at scale | measuring all of the above |
| L89 | Capstone: ship a RAG service | put it in production |

Chunking comes **first** on purpose. It's cheap, it's deterministic, and it caps
everything after it. Fixing retrieval or reranking on top of bad chunks is polishing
a floor with a hole in it.

In [ ]:
# In Colab this installs into the runtime. Locally it's a no-op if already present.
!pip install scikit-learn numpy -q

import re
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# This whole lesson is fully OFFLINE and deterministic. No API key, no downloads.
# We reuse the SAME TF-IDF retriever from L83 so that the ONLY thing changing between
# experiments is how we cut the documents into chunks.
print("Ready. Everything below is deterministic and offline.")

## §0 · Recap: the cliffhanger from Lesson 83

In L83 the pipeline was `load -> chunk -> embed -> index -> retrieve -> generate`, and
the final sweep looked like this (same corpus, same retriever, same 8 questions):

```
max_chars   nchunks   hit@1
   60          ~40      lower
  180           24      higher
   ∞ (1 chunk)   8      different again
```

We had no principled reason to prefer one number. "Try a few and pick the best on your
eval" is honest advice, but it's not *understanding*. To understand it we need to see
the two distinct ways a bad cut destroys a fact:

1. **Fragmentation** — the fact gets split across a boundary, so no single chunk holds
   it whole (or the answer gets divorced from the words the question matches on).
2. **Dilution** — the chunk is so big that the one relevant sentence is buried among
   many irrelevant ones, and its signal gets averaged away.

Good chunking is the art of avoiding *both* at once.

## §1 · The corpus and the eval (unchanged from L83)

Same 8 fictional-SaaS "Nimbus" docs, same labelled questions. We keep them byte-for-byte
so today's numbers are directly comparable to last lesson's. The only change: the
`expect` strings are now the **full answer phrase** (e.g. `"600 requests per minute"`,
not just `"600"`) so our stricter metric below can tell whether a chunk truly preserved
the fact end-to-end.

In [ ]:
DOCS = {
 "refunds": ("Nimbus Refund Policy. Customers on the monthly plan may request a full refund "
   "within 14 days of any charge. Annual plans are refundable on a prorated basis for the "
   "remaining unused months. Refunds are issued to the original payment method and take 5 to "
   "10 business days to appear. One-time setup fees are non-refundable."),
 "ratelimits": ("Nimbus API Rate Limits. The Free tier allows 60 requests per minute. The Pro "
   "tier allows 600 requests per minute. The Enterprise tier allows 6000 requests per minute. "
   "Exceeding your limit returns HTTP 429. Each 429 response includes a Retry-After header "
   "telling you how many seconds to wait before retrying."),
 "sso": ("Nimbus Single Sign-On. SSO is available on the Enterprise plan only. We support SAML "
   "2.0 and OIDC. To configure SAML, an administrator uploads the identity provider metadata XML "
   "in the Security settings page. Just-in-time user provisioning is enabled by default so new "
   "users are created on first login."),
 "retention": ("Nimbus Data Retention. Application logs are retained for 30 days. Deleted "
   "projects are held in a recoverable trash state for 90 days before permanent deletion. "
   "Customers on the Enterprise plan can configure a custom retention window of up to 7 years "
   "for compliance."),
 "security": ("Nimbus Security. All customer data is encrypted at rest using AES-256 and in "
   "transit using TLS 1.3. Nimbus is SOC 2 Type II certified. Access to production systems "
   "requires hardware security keys. We run third-party penetration tests twice per year."),
 "credentials": ("Nimbus Account Access. If you are locked out, use the credential recovery flow "
   "on the sign-in page: enter your email and we send a one-time link that lets you set a new "
   "secret. Links expire after 30 minutes. Enabling two-factor authentication is strongly "
   "recommended for all accounts."),
 "pricing": ("Nimbus Pricing. The Free tier costs nothing and includes one project. The Pro tier "
   "costs 49 dollars per month and includes ten projects. The Enterprise tier is custom-priced "
   "and includes unlimited projects, SSO, and a dedicated support manager."),
 "support": ("Nimbus Support SLAs. Free tier support is community-only. Pro tier guarantees a "
   "first response within one business day. Enterprise tier guarantees a first response within "
   "one hour for urgent issues, twenty-four hours a day, seven days a week."),
}

# Each question names the doc that holds the answer (gold) and the FULL answer phrase we
# expect to survive intact inside the retrieved chunk.
EVAL = [
 {"q":"how many days to get a refund on a monthly plan","gold":"refunds","expect":"14 days"},
 {"q":"what is the Pro tier rate limit","gold":"ratelimits","expect":"600 requests per minute"},
 {"q":"which plans include single sign-on","gold":"sso","expect":"Enterprise"},
 {"q":"how long are deleted projects recoverable","gold":"retention","expect":"90 days"},
 {"q":"what encryption is used for data at rest","gold":"security","expect":"AES-256"},
 {"q":"how much does the Pro plan cost per month","gold":"pricing","expect":"49 dollars"},
 {"q":"how fast does Enterprise support respond to urgent issues","gold":"support","expect":"one hour"},
 {"q":"what is the maximum custom retention window for compliance","gold":"retention","expect":"7 years"},
]
print(f"{len(DOCS)} documents, {len(EVAL)} labelled questions.")

## §2 · Five ways to cut a document

Every strategy below has the **same signature** — `(doc_id, text) -> list of chunk
dicts` — so they are drop-in swappable and we can measure them apples-to-apples. A chunk
dict is just `{"id": "docname#3", "doc": "docname", "text": "..."}`.

| Strategy | How it decides where to cut | Cost | Typical failure |
|----------|-----------------------------|------|-----------------|
| **fixed-char** | every N characters, blindly | trivial | cuts mid-word / mid-fact |
| **fixed-char + overlap** | every N chars, but each chunk repeats the last M chars of the previous | trivial | wastes space; still blind |
| **sentence-aware** | pack whole sentences up to a budget (this is L83's chunker) | cheap | can't split *within* a long sentence |
| **recursive** | try the biggest natural separator first (paragraph → sentence → space → char) that fits the budget | cheap | needs a good separator hierarchy |
| **semantic** | start a new chunk when the topic *shifts* (adjacent-sentence similarity drops) | needs embeddings | over/under-splits if the threshold is wrong |

The first four are **structural** — they look at characters and punctuation. The last is
**semantic** — it looks at meaning. Intuitively the semantic one should win. Hold that
thought; the numbers have a surprise.

In [ ]:
# --- shared helper: split text into sentences on . ! ? boundaries ---
def sent_split(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

# --- Strategy 1 & 2: fixed-size character windows, with optional overlap ---
def chunk_fixed(doc_id, text, size=180, overlap=0):
    # Slice every `size` characters. `overlap` repeats the tail of the previous chunk so a
    # fact sitting on a boundary appears whole in at least one chunk. Blind to word/sentence.
    out, i, k = [], 0, 0
    step = max(1, size - overlap)
    while i < len(text):
        out.append({"id": f"{doc_id}#{k}", "doc": doc_id, "text": text[i:i+size]})
        k += 1
        i += step
    return out

# --- Strategy 3: sentence-aware packing (L83's chunker, kept verbatim) ---
def chunk_sentence(doc_id, text, max_chars=180, overlap_sents=1):
    # Pack whole sentences up to max_chars; never split inside a sentence; carry 1 sentence over.
    sents = sent_split(text)
    out, idx, k = [], 0, 0
    while k < len(sents):
        buf, length, start = [], 0, k
        while k < len(sents) and (length + len(sents[k]) <= max_chars or not buf):
            buf.append(sents[k]); length += len(sents[k]) + 1; k += 1
        out.append({"id": f"{doc_id}#{idx}", "doc": doc_id, "text": " ".join(buf)})
        idx += 1
        if overlap_sents and k < len(sents):
            k = max(k - overlap_sents, start + 1)
    return out

print("fixed + sentence chunkers defined.")

In [ ]:
# --- Strategy 4: recursive character splitting ---
# The industry-standard splitter (LangChain's default). Try to break on the BIGGEST natural
# separator first; if a piece is still too big, recurse into the next-smaller separator.
# The ordered hierarchy means we break at paragraph boundaries before sentences, sentences
# before spaces, and only ever hack mid-word as a last resort.
SEPARATORS = ["\n\n", "\n", ". ", " ", ""]

def _recurse(text, size, seps):
    if len(text) <= size:
        return [text]
    if not seps:                                        # no separators left: hard char cut
        return [text[i:i+size] for i in range(0, len(text), size)]
    sep = seps[0]
    if sep == "":
        return [text[i:i+size] for i in range(0, len(text), size)]
    parts = text.split(sep)
    chunks, cur = [], ""
    for p in parts:
        cand = p if cur == "" else cur + sep + p
        if len(cand) <= size:
            cur = cand                                  # this part still fits: keep packing
        else:
            if cur:
                chunks.append(cur)                      # flush what we had
            if len(p) > size:
                chunks.extend(_recurse(p, size, seps[1:]))  # a single part is too big: recurse
                cur = ""
            else:
                cur = p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_recursive(doc_id, text, size=180):
    out = []
    for piece in _recurse(text, size, SEPARATORS):
        piece = piece.strip()
        if piece:
            out.append({"id": f"{doc_id}#{len(out)}", "doc": doc_id, "text": piece})
    return out

# --- Strategy 5: semantic chunking ---
# Start a NEW chunk whenever the meaning shifts: measure similarity between each sentence and
# the one before it; a low similarity means the topic changed, so cut there. (We use TF-IDF
# over the doc's own sentences as a cheap stand-in for a real embedding model.)
def chunk_semantic(doc_id, text, max_chars=220, sim_threshold=0.10):
    sents = sent_split(text)
    if len(sents) <= 1:
        return [{"id": f"{doc_id}#0", "doc": doc_id, "text": text}]
    vec = TfidfVectorizer().fit(sents)
    S = vec.transform(sents)
    out, buf, idx, length = [], [sents[0]], 0, len(sents[0])
    for k in range(1, len(sents)):
        sim = float(cosine_similarity(S[k], S[k-1])[0][0])
        if sim < sim_threshold or length + len(sents[k]) > max_chars:
            out.append({"id": f"{doc_id}#{idx}", "doc": doc_id, "text": " ".join(buf)})
            idx += 1; buf = [sents[k]]; length = len(sents[k])
        else:
            buf.append(sents[k]); length += len(sents[k]) + 1
    out.append({"id": f"{doc_id}#{idx}", "doc": doc_id, "text": " ".join(buf)})
    return out

print("recursive + semantic chunkers defined. All five strategies ready.")

## §3 · The measuring stick: two metrics, not one

We keep L83's `VectorIndex` (TF-IDF + cosine) untouched, and score each strategy on **two**
metrics — because they measure different things and only together tell the truth:

- **`hit@1 (doc)`** — is the top retrieved chunk from the *right document*? This is the
  forgiving, retrieval-level metric. Bad chunking can still score well here because *any*
  chunk of the right doc counts.
- **`answer_in_top1`** — does the top retrieved chunk **actually contain the full answer
  phrase**? This is the strict, chunk-level metric. This is where fragmentation shows up:
  you can retrieve the right document but the exact fact got sliced away or divorced from
  the query's words.

`answer_in_top1` is the honest scoreboard for chunking. Watch it closely.

In [ ]:
class VectorIndex:
    # Identical to L83: TF-IDF vectors + cosine similarity. The retriever is a CONSTANT today.
    def __init__(self, chunks):
        self.chunks = chunks
        self.vec = TfidfVectorizer(stop_words="english")
        self.M = self.vec.fit_transform([c["text"] for c in chunks])
    def search(self, query, k=5):
        sims = cosine_similarity(self.vec.transform([query]), self.M)[0]
        order = np.argsort(-sims)[:k]
        return [(self.chunks[i], float(sims[i])) for i in order]

def build_chunks(fn, **kw):
    chunks = []
    for d, t in DOCS.items():
        chunks.extend(fn(d, t, **kw))
    return chunks

def hit_at_doc(rids, gold, n):
    return any(cid.split('#')[0] == gold for cid in rids[:n])

def score(chunks, k=5):
    idx = VectorIndex(chunks)
    hit1 = ans1 = 0
    for e in EVAL:
        r = idx.search(e["q"], k=k)
        rids = [c["id"] for c, _ in r]
        hit1 += hit_at_doc(rids, e["gold"], 1)
        top_text = r[0][0]["text"]
        ans1 += (e["expect"].lower() in top_text.lower())
    n = len(EVAL)
    return {"nchunks": len(chunks), "hit@1": hit1/n, "answer_in_top1": ans1/n}

# Build every strategy once and keep them for reuse below.
STRATS = {
 "fixed-char (no overlap)": build_chunks(chunk_fixed, size=180, overlap=0),
 "fixed-char (overlap 60)": build_chunks(chunk_fixed, size=180, overlap=60),
 "sentence-aware (L83)":    build_chunks(chunk_sentence, max_chars=180),
 "recursive":               build_chunks(chunk_recursive, size=180),
 "semantic (untuned)":      build_chunks(chunk_semantic, max_chars=220),
}

print(f"{'strategy':26}{'nchunks':>9}{'hit@1(doc)':>12}{'answer_in_top1':>16}")
print("-"*63)
RESULTS = {}
for name, ch in STRATS.items():
    s = score(ch); RESULTS[name] = s
    print(f"{name:26}{s['nchunks']:>9}{s['hit@1']:>12.2f}{s['answer_in_top1']:>16.2f}")

### Read that table slowly — there are three surprises

1. **`hit@1(doc)` barely moves, but `answer_in_top1` moves a lot.** Every strategy finds the
   right *document* most of the time; they differ on whether the top chunk still holds the
   *actual fact*. If you'd only measured document-level retrieval (as many tutorials do) you
   would have concluded "chunking doesn't matter." It matters — you just need the right ruler.

2. **Naive fixed-char loses a fact (`answer_in_top1` < 1.00), and BOTH overlap and recursive
   fix it — by completely different mechanisms.** Overlap heals the wound after the fact by
   duplicating the seam; recursive avoids the wound by never cutting at a bad place. We dissect
   the exact failure in §4.

3. **Semantic — the "smartest" strategy — scores WORST here.** Untuned, with `max_chars=220`
   over this tiny corpus, it over-splits into ~30 shards and fragments facts. This is the
   lesson people skip: *a fancier algorithm with a wrong knob loses to a dumb one with a right
   knob.* Semantic chunking earns its keep on long, topically-mixed documents (a 40-page
   handbook), not 4-sentence FAQ entries. We'll see exactly why in §5.

## §4 · Anatomy of one broken cut

The naive `fixed-char (no overlap)` strategy misses the question *"how fast does Enterprise
support respond to urgent issues"* (answer: **one hour**). Let's see the crime scene. The
support document is one paragraph; a blind 180-character cut lands right in the middle of the
Enterprise SLA sentence.

In [ ]:
support_chunks = [c for c in STRATS["fixed-char (no overlap)"] if c["doc"] == "support"]
print("fixed-char cut the SUPPORT doc into:")
for c in support_chunks:
    print(f"  [{c['id']}] {c['text']!r}\n")

# The retriever's answer for the urgent-issues question:
idx_fixed = VectorIndex(STRATS["fixed-char (no overlap)"])
q = "how fast does Enterprise support respond to urgent issues"
top = idx_fixed.search(q, k=1)[0][0]
print("Query:", q)
print("Top chunk retrieved:", top["id"])
print("Does it contain 'one hour'? ->", "one hour" in top["text"].lower())

See what happened? The 180-char boundary fell **between `one hour` and `for urgent issues`**.
So:

- Chunk `support#0` ends with `...within one hour` — it has the **answer** but not the words
  the query leans on.
- Chunk `support#1` starts with `for urgent issues, twenty-four hours a day...` — it has the
  **query-bait words** (`urgent issues`) but *not the answer*.

The retriever, matching on `urgent issues`, confidently returns `support#1` — the half **without
the answer**. The fact wasn't deleted; it was *divorced from the words that find it*. That is
fragmentation in its most insidious form, and no downstream reranker can fix it, because the
answer and the query-words now live in different chunks.

Now watch overlap and recursive heal it:

In [ ]:
for name in ["fixed-char (overlap 60)", "recursive"]:
    idx = VectorIndex(STRATS[name])
    top = idx.search(q, k=1)[0][0]
    print(f"{name:26} -> top {top['id']:12} contains 'one hour'? {'one hour' in top['text'].lower()}")
    print(f"{'':26}    {top['text'][:110]!r}\n")

# Overlap works because the repeated 60-char seam puts 'one hour for urgent issues' whole
# into one chunk. Recursive works because it splits on '. ' and never breaks inside a sentence.

**Overlap is cheap insurance against boundary-straddling facts.** By repeating the tail of
each chunk at the head of the next, any fact shorter than the overlap window is guaranteed to
appear *whole* somewhere. The cost: redundant text inflates your index (more vectors, more
storage, and the same fact can now occupy multiple retrieval slots). A common production
default is **10–20% overlap**. Recursive/structure-aware splitting attacks the same problem
from the other side — it tries never to create the bad boundary in the first place — which is
why in practice teams often use *both*: recursive splitting **with** a small overlap.

## §5 · The size knob: fragmentation vs dilution

Chunk size trades off the two failure modes from §0. Sweep it on the recursive splitter and
watch `answer_in_top1`:

In [ ]:
print(f"{'size':>8}{'nchunks':>10}{'answer_in_top1':>16}")
print("-"*34)
for sz in (40, 80, 120, 180, 260, 100000):
    ch = build_chunks(chunk_recursive, size=sz)
    print(f"{sz:>8}{len(ch):>10}{score(ch)['answer_in_top1']:>16.2f}")
print("\nToo small -> facts fragment (answer_in_top1 collapses).")
print("Larger    -> facts stay whole on THIS short corpus, so it plateaus at 1.00.")
print("On real long docs the top end falls too, as one big chunk DILUTES the relevant sentence.")

On this deliberately tiny corpus the curve rises from a fragmented mess at `size=40` and then
**plateaus** — because the documents are only a few sentences long, so even "one chunk per doc"
doesn't dilute much. On real documents the curve is a **hump**: it climbs out of fragmentation,
peaks, then falls as oversized chunks bury the answer among noise. The peak is corpus-specific,
which is the whole reason L83's FM4 existed — *you find the peak by measuring on your own eval,
exactly like this, not by guessing.* Chunking has no universal best size; it has a best size
**for your data and your questions.**

## §6 · Proof that chunking sets the ceiling

Here is the claim that makes chunking the highest-leverage stage, made concrete. Cut the
rate-limits doc into tiny 40-char pieces and ask: does the fact **"600 requests per minute"**
survive *anywhere*?

In [ ]:
tiny = chunk_recursive("ratelimits", DOCS["ratelimits"], size=40)
print("40-char chunks of the rate-limits doc:")
for c in tiny[:6]:
    print("  ", repr(c["text"]))

fact = "600 requests per minute"
survives = any(fact in c["text"] for c in tiny)
print(f"\nIs '{fact}' present WHOLE in ANY chunk? -> {survives}")
print("It was shattered into 'The Pro tier allows 600' + 'requests per minute'.")
print("A better embedding model (L85) reorders chunks. A reranker (L86) reorders chunks.")
print("Neither can return a fact that no chunk contains. The ceiling was set at chunk time.")

That's the argument in one cell. **The retriever, the reranker, and the LLM all operate on
the set of chunks you produced — they can reorder them, score them, and read them, but they
can never reconstitute a fact your splitter tore in half.** Chunking is the one stage whose
mistakes are *unrecoverable* downstream. Spend your first hour of any RAG project here.

## §7 · So when *is* semantic chunking worth it?

Semantic chunking lost today because our docs are already short and single-topic — there was
no topic drift to detect, so measuring it only added a knob to get wrong. It shines when
structural splitting is blind to meaning:

- **Long, heterogeneous documents** — a policy handbook where "refunds," "SSO," and "security"
  sit in one 30-page file. A fixed/recursive splitter cuts every N chars regardless of subject;
  semantic chunking puts each topic in its own chunk, so retrieval isn't polluted by neighbours.
- **Transcripts and chat logs** where speaker turns and topic shifts don't line up with
  punctuation.
- **Documents with no clean separators** — no headings, no paragraphs, just a wall of prose.

The practical hierarchy most teams settle on: **structure-aware (recursive) splitting as the
default**, add **overlap** as insurance, and reach for **semantic chunking only when your docs
are long and mixed enough that topic boundaries ≠ character boundaries.** Always validate the
choice on your own eval — precisely the harness you're building across this phase.

## §8 · Ten production chunking pitfalls

| # | Pitfall | Why it bites |
|---|---------|--------------|
| 1 | Blind fixed-char splitting | slices facts and words mid-stream |
| 2 | Zero overlap | boundary-straddling facts vanish from every chunk |
| 3 | Too-small chunks | fragmentation; the answer needs 2 chunks to be whole |
| 4 | Too-large chunks | dilution; the relevant sentence drowns in noise |
| 5 | One global chunk size for all doc types | code, tables, prose want different sizes |
| 6 | Ignoring document structure | throwing away headings/markdown/tables that mark real boundaries |
| 7 | Chunking away tables & code as prose | rows/functions get sliced into nonsense |
| 8 | No metadata on chunks | can't cite, filter, or trace which doc/section an answer came from |
| 9 | Re-chunking with a different size but not re-indexing | index and chunker silently disagree |
| 10 | Never measuring chunking on a real eval | you "tune" blind and ship FM4 to prod |

Notice pitfalls 1–4 are exactly what the two metrics in §3 catch, and pitfall 10 is the meta-sin
this entire phase exists to cure.

## §9 · Verification — every claim in this lesson, checked

In [ ]:
checks = []
def check(name, cond):
    checks.append(bool(cond))
    print(("PASS " if cond else "FAIL ") + name)

R = {name: score(ch) for name, ch in STRATS.items()}
fixed  = R["fixed-char (no overlap)"]["answer_in_top1"]
overlap= R["fixed-char (overlap 60)"]["answer_in_top1"]
sent   = R["sentence-aware (L83)"]["answer_in_top1"]
recur  = R["recursive"]["answer_in_top1"]
seman  = R["semantic (untuned)"]["answer_in_top1"]

# All five strategies still find the right DOCUMENT most of the time (retrieval is the constant).
check("every strategy keeps hit@1(doc) >= 0.85",
      all(R[n]["hit@1"] >= 0.85 for n in R))

# Naive fixed-char loses at least one FACT to a boundary.
check("fixed-char (no overlap) drops a fact (answer_in_top1 < 1.0)", fixed < 1.0)

# Overlap recovers it, by duplicating the seam.
check("overlap recovers the lost fact (>= fixed, and == 1.0)", overlap >= fixed and overlap == 1.0)

# Structure-aware recursive is best-in-class and never worse than sentence-aware.
check("recursive is perfect and >= sentence-aware", recur == 1.0 and recur >= sent)

# The cautionary tale: untuned semantic UNDERPERFORMS the dumb structural splitter.
check("untuned semantic underperforms recursive (fancier != better)", seman < recur)

# The exact boundary crime: fixed-char returns the support half WITHOUT the answer...
idx_f = VectorIndex(STRATS["fixed-char (no overlap)"])
top_fixed = idx_f.search("how fast does Enterprise support respond to urgent issues", 1)[0][0]["text"]
check("fixed-char top chunk for the SLA query lacks 'one hour'",
      "one hour" not in top_fixed.lower())
# ...but overlap's top chunk for the same query HAS it.
idx_o = VectorIndex(STRATS["fixed-char (overlap 60)"])
top_ov = idx_o.search("how fast does Enterprise support respond to urgent issues", 1)[0][0]["text"]
check("overlap top chunk for the SLA query contains 'one hour'",
      "one hour" in top_ov.lower())

# Size is non-monotonic on the fragmentation side: tiny chunks are much worse than mid-size.
tiny_score = score(build_chunks(chunk_recursive, size=40))["answer_in_top1"]
mid_score  = score(build_chunks(chunk_recursive, size=180))["answer_in_top1"]
check("tiny chunks fragment facts (size 40 << size 180)", tiny_score < mid_score)

# The ceiling proof: a fact shattered by tiny chunks exists in NO chunk -> unrecoverable.
tiny_rl = chunk_recursive("ratelimits", DOCS["ratelimits"], size=40)
check("a shattered fact appears in ZERO chunks (chunking = the ceiling)",
      not any("600 requests per minute" in c["text"] for c in tiny_rl))

print(f"\n{sum(checks)}/{len(checks)} checks passed")
assert sum(checks) == len(checks), "some checks failed"
print("ALL CHECKS PASSED")

## Summary, homework, and what's next

**What you learned**

| Concept | The takeaway |
|---------|--------------|
| Chunking = the ceiling | downstream stages reorder chunks; none can rebuild a fact you split |
| Two metrics | `hit@1(doc)` is forgiving; `answer_in_top1` is the honest chunking ruler |
| Fragmentation | a cut can divorce the answer from the words the query matches on |
| Overlap | cheap insurance — repeat the seam so straddling facts stay whole (costs index bloat) |
| Recursive splitting | the sensible default — break on the biggest natural separator that fits |
| Semantic chunking | powerful on long mixed docs, but a wrong knob loses to a dumb-but-tuned splitter |
| Size = fragmentation vs dilution | non-monotonic; the peak is corpus-specific, found by measuring |

**Homework**

1. Add 4 new questions to `EVAL` whose answers straddle a sentence boundary; find the fixed-char
   size that breaks the most of them, then show overlap and recursive both recover them.
2. Implement `chunk_recursive_with_overlap` (recursive split, then add a sliding N-char overlap)
   and measure whether it beats plain recursive on your expanded eval.
3. Swap the semantic chunker's TF-IDF for real `sentence-transformers` (`all-MiniLM-L6-v2`)
   adjacent-sentence similarity, then write one long synthetic doc that mixes 3 topics and show
   semantic chunking finally *wins* on it.
4. Attach metadata to each chunk (`{"section": ..., "source": ...}`) and make `answer()` cite the
   `source` — you'll need this for grounding in L87.
5. Turn the §5 sweep into a tiny `pick_chunk_size(eval, strategy)` function that returns the size
   maximizing `answer_in_top1`. This is FM4, solved and automated.

**Next lesson — L85: Dense & Hybrid Retrieval.** We stop letting chunking take all the blame.
Even with perfect chunks, TF-IDF is *lexically blind*: L83's "how do I reset my password" scored
**0.000** because the doc says "credential recovery / set a new secret" and shares zero words with
the query. That's **Failure Mode #2**, the semantic gap. L85 replaces the lexical retriever with
dense embeddings (and a hybrid of both) so meaning — not just shared words — drives retrieval.